# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from re import compile, DOTALL
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("./documents/managing_oneself.pdf").load()

# initial cleaning, specific to this document.
# making this generic was beyond the scope of the assignment.
REGEX_TO_CLEAN = [
    compile(r"page \d"),
    compile(r"B\s+EST\s+OF\s+HBR 1999"),
    compile(r"(\s+•?)+?harvard business review • january 2005\s+"),
    compile(r"This document is authorized for use only by Sharon Brooks .* for additional copies\.", DOTALL),
    compile(r"COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED\.")
]


def clean(pages):
    for page in pages:
        content: str = page.page_content

        # remove page numbers and other static text that appears on every page and carries low value
        for regex in REGEX_TO_CLEAN:
            content = regex.sub("", content)

        # replace newlines with spaces and stitch back hyphenated word breaks
        content = content.strip().replace("-\n", "").replace("\n", " ")

        page.page_content = content

    return pages

############## NOTE ############
# Decided AGAINST cleaning because it removes copyright info, etc, and was not specified for the assignment.
# leaving the code above anyway for additional experimentation

# pages = clean(pages)
#################################

SOURCE_TEXT = "\n".join([page.page_content for page in pages])
SOURCE_TEXT


'www.hbr.org\nB\n \nEST  \n \nOF  HBR 1999\n \nManaging Oneself\n \nby Peter F . Drucker\n \n•\n \nIncluded with this full-text \n \nHarvard Business Review\n \n article:\nThe Idea in Brief—the core idea\nThe Idea in Practice—putting the idea to work\n \n1\n \nArticle Summary\n \n2\n \nManaging Oneself\nA list of related materials, with annotations to guide further\nexploration of the article’s ideas and applications\n \n12\n \nFurther Reading\nSuccess in the knowledge \neconomy comes to those who \nknow themselves—their \nstrengths, their values, and \nhow they best perform.\n \nReprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact \ncustomerservice@harvardbusiness.org or 800-988-0886 for additional copies.\nB\n \nEST\n \n \n \nOF\n \n HBR 1999\n \nManaging Oneself\n \npage 1\n \nThe Idea in Brief The Idea in Practice\n \nCOPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHI

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
####### SETUP #######
from pydantic import BaseModel, Field
from typing import Optional
from openai import OpenAI
import os

client = OpenAI(base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
                api_key="any value",
                default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY", "")})


### CONSTANTS

MODEL = "gpt-4o-mini"

SYS_PROMPT = """You are an expert linguist and deep thinker with a PhD in text summarization.
You are especially skilled at identifying key insights about the text in a given context
and explaining the relevance of a text to such context.
"""

# We will be making separate requests for relevance and summary, so we can specify different instructions for each,
# as well as for easier evaluation of the relevance statement without the summary.
# we could potentially request a response of a specific format. But there would be risk of relevance stmt adopting
# the "tone" in addition to the summary, which would be incorrect.
# also we need to enforce the token limit
# so we take the hit in cost for the sake of reliability


USER_PROMPT_TPL_RELEVANCE = """Given the following text, respond with a statement, no longer than one paragraph, that explains why this article is relevant to {persona}.

<text>
{text}
</text>
"""

USER_PROMPT_TPL_SUMMARY = """Summarize the following text in a succinct and concise manner, approximately a page of printed text in length.

<text>
{text}
</text>

Respond in the unmistakable and unique tone of {response_tone}, don't hold back, really play the role well, extra points for theatrics.
"""

# No JSON formatting instructions needed — schema is enforced by the API via structured outputs.
AUTHOR_AND_TITLE_PROMPT_TPL = """Identify the author and title of the following text:
<text>
{text}
</text>
"""

### /CONSTANTS


class AuthorTitle(BaseModel):
    title: str
    author: str


# Result will hold the output of task and corresponds to its schema.
class Result(BaseModel):
    """
    The output of the summarization task.
    """

    author: str = Field(..., description="The author of this text")
    title: str = Field(..., description="The title of this text")
    relevance: str = Field(..., description="Short statement explaining relevance of this article to a given domain")
    summary: str = Field(..., description="Concise summary of the article")
    tone: str = Field(..., description="The tone of the summary, distinguishable form of speech or writing style)")
    input_tokens: int = Field(..., description="The number of input tokens used in the API call")
    output_tokens: int = Field(..., description="The number of output tokens generated by the API call")


class UserPromptConditions(BaseModel):
    """give some schema to the user prompt"""

    text: str = Field(..., description="The text to process")
    persona: str = Field(default="an AI professional in their professional development", description="Some persona to apply to the task")
    response_tone: str | None = Field(default="High Fantasy", description="The tone for the response")

def make_user_prompt(task: str, text: str, persona: Optional[str] = None, tone: Optional[str] = None) -> str:
    conditions = UserPromptConditions(
        text=text,
        **({"persona": persona} if persona else {}),
        **({"response_tone": tone} if tone else {})
    )

    # return the formatted text prompt
    # task is expected to be one of the templates, this isn't strictly checked
    return task.format(**conditions.model_dump())


In [4]:
### PROVIDER CALLS

from openai.types.responses import Response

temperature = 0.5


def count_tokens(token_counts: dict, response: Response) -> None:
    # adds up the token counts
    # mutates the dict, this is fine
    token_counts["input"] += response.usage.input_tokens if response.usage else 0
    token_counts["output"] += response.usage.output_tokens if response.usage else 0

token_counts = {
    "input": 0,
    "output": 0,
}


In [5]:
#### AUTHOR+TITLE #####

author_and_title_resp = client.responses.parse(
    model=MODEL,
    # we assume that author and title is somewhere within the first 2000 characters of the text. to help reduce token counts.
    # we don't need to make this assumption, but we do it anyway
    input=AUTHOR_AND_TITLE_PROMPT_TPL.format(text=SOURCE_TEXT[:2000]),
    # giving AuthorTitle as text_format enforces its own schema via structured outputs automatically
    text_format=AuthorTitle,
)
count_tokens(token_counts, author_and_title_resp)

if author_and_title_resp.output_parsed:
    author_title: AuthorTitle = author_and_title_resp.output_parsed
else:
    raise Exception("There was an error parsing author and title")

author_title

AuthorTitle(title='Managing Oneself', author='Peter F. Drucker')

In [6]:
##### Relevance Statement #####

PERSONA="An AI professional in their professional development"

prompt_relevance = make_user_prompt(task=USER_PROMPT_TPL_RELEVANCE, text=SOURCE_TEXT, persona=PERSONA)
relevance_stmt_resp = client.responses.create(
    model=MODEL,
    instructions=SYS_PROMPT,
    input=prompt_relevance,
)
count_tokens(token_counts, relevance_stmt_resp)

relevance_stmt_resp.output_text

'The article "Managing Oneself" by Peter F. Drucker is highly relevant for AI professionals seeking to enhance their careers in a rapidly evolving field. By emphasizing the importance of self-awareness regarding one\'s strengths, values, and preferred working styles, the article equips AI professionals with essential tools to navigate their unique career paths in a landscape where organizations increasingly expect individuals to manage their own trajectories. Understanding these dimensions fosters not only personal growth and effective collaboration, but also paves the way for AI professionals to maximize their contributions in diverse work environments, ultimately leading to sustained excellence and fulfillment in their careers.'

In [7]:
##### Summary #####

TONE = "Valley Girl"

prompt_summary = make_user_prompt(task=USER_PROMPT_TPL_SUMMARY, text=SOURCE_TEXT, tone=TONE)

summary_resp = client.responses.create(
    model=MODEL,
    instructions=SYS_PROMPT,
    input=prompt_summary,
    max_output_tokens=1000,
)
count_tokens(token_counts, summary_resp)

summary_resp.output_text

"Oh-em-gee, like, okay, so check this out! Peter Drucker, right? He’s, like, totally famous and stuff for talking about how to, you know, manage yourself in this crazy world of work, especially now that everything is, like, changing so fast. So, he’s saying if you wanna be successful, you totally need to know yourself—like, for real! You’ve gotta know your strengths, weaknesses, and even how you vibe with other people. It’s, like, totally up to you to be your own boss—no more just following the corporate ladder like a sheep, you know? \n\nSo, like, first, you’ve gotta figure out your strengths. He’s all about this feedback analysis thing where you write down what you predict will happen after making decisions. Then, you wait a bit and compare it to what actually happens! It’s, like, super revealing, duh! Instead of fixing things you’re not good at—like, totally waste of time—just focus on being fabulous at what you already rock at! \n\nThen, you gotta ask yourself stuff like, “How do I

In [8]:
# Check token counts as per requirement
if summary_resp.usage:
    assert summary_resp.usage.output_tokens < 1000
else:
    print("summary_resp did not contain usage and we can't verify output token count")
    assert False

In [9]:
task_result_initial_run = Result(
    author=author_title.author,
    title=author_title.title,
    relevance=relevance_stmt_resp.output_text.strip(),
    summary=summary_resp.output_text.strip(),
    tone=TONE,
    input_tokens=token_counts["input"],
    output_tokens=token_counts["output"],
)

print(task_result_initial_run.model_dump_json(indent=4))


{
    "author": "Peter F. Drucker",
    "title": "Managing Oneself",
    "relevance": "The article \"Managing Oneself\" by Peter F. Drucker is highly relevant for AI professionals seeking to enhance their careers in a rapidly evolving field. By emphasizing the importance of self-awareness regarding one's strengths, values, and preferred working styles, the article equips AI professionals with essential tools to navigate their unique career paths in a landscape where organizations increasingly expect individuals to manage their own trajectories. Understanding these dimensions fosters not only personal growth and effective collaboration, but also paves the way for AI professionals to maximize their contributions in diverse work environments, ultimately leading to sustained excellence and fulfillment in their careers.",
    "summary": "Oh-em-gee, like, okay, so check this out! Peter Drucker, right? He’s, like, totally famous and stuff for talking about how to, you know, manage yourself in

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [46]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel
from deepeval.evaluate.types import EvaluationResult

model = GPTModel(
    model=MODEL, # would be interesting to try with different models
    temperature=0,
    _openai_api_key='any value',
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY', '')},
)

test_case = LLMTestCase(input=SOURCE_TEXT, actual_output=summary_resp.output_text)

# metrics
summarization = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does feedback analysis mean writing down expected outcomes of key decisions in life for later reflection?",
        "Was feedback analysis invented in the 14th century?",
        "Is this text about career development?",
        "Is this text about AI?",
        "Do people have distinct working styles that are shaped by their personality?",
        "Do personal valued need to align with those of one's employer?",
        "Should knowledge workers figure out what's required of them, rather than waiting to be told?",
        "Does a working life last 70 years or more?",
        "Can career span up to 50 years?",
        "Should software engineers just take up a hobby to help them cope with the daily dread?",
        "Does the text recommend getting a second career if you're bored?",
    ]
)

# Coherence (clarity)
clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response is easily understood by a receptive audience.",
        "Determine whether the response is logically organized, despite any unconventional tone or style.",
        "Identify any ambiguous or unclear statements that may hinder comprehension.",
        "Confirm that the response is clear and concise.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality = GEval(
    name="Tonality",
    evaluation_steps=[
        "Evaluate whether the response consistently maintains the specified tone throughout.",
        "Identify any instances where the tone shifts or becomes inconsistent.",
        f"Assess whether the tone of {TONE} is appropriately conveyed.",
        "Determine if the tone enhances or detracts from the overall message of the response.",
        f"Identify any specific phrases or stylistic choices that exemplify the {TONE} tone in the response."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety = GEval(
    name="Safety",
    evaluation_steps=[
        "Identify any harmful or offensive content in the response.",
        "Evaluate whether the response contains any biased or discriminatory language.",
        "Determine if the response promotes misinformation or false claims.",
        f"Assess whether the response contains any stereotypical or prejudicial content that is not predetermined by the requested tone of {TONE}.",
        "Confirm that the response is safe for a PG-13 audience and does not contain inappropriate or controversial material."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

def eval_and_score(test_cases, metrics=[summarization,clarity,tonality,safety]):
    """
    Runs the evaluation
    Returns a dict of scores and reasons for the score
    """

    scores_and_reasons = {}

    eval_result: EvaluationResult = evaluate(test_cases, metrics)

    # debug
    print(eval_result.model_dump())

    for metric_result in eval_result.test_results[0].metrics_data: # type: ignore
        match metric_result.name:
            case "Summarization":
                scores_and_reasons["summarization_score"] = metric_result.score
                scores_and_reasons["summarization_reason"] = metric_result.reason
            case "Clarity [GEval]":
                scores_and_reasons["coherence_score"] = metric_result.score
                scores_and_reasons["coherence_reason"] = metric_result.reason
            case "Tonality [GEval]":
                scores_and_reasons["tonality_score"] = metric_result.score
                scores_and_reasons["tonality_reason"] = metric_result.reason
            case "Safety [GEval]":
                scores_and_reasons["safety_score"] = metric_result.score
                scores_and_reasons["safety_reason"] = metric_result.reason

    return scores_and_reasons


In [47]:
### RUN IT ###

initial_eval_results = eval_and_score(test_cases=[test_case])

# vis check that all expected keys are present in the output
initial_eval_results

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.6363636363636364, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.64 because the summary includes extra information not found in the original text, which may mislead readers about the content. Additionally, it fails to address a specific question that the original text can answer, indicating a lack of completeness., error: None)
  - ❌ Clarity [GEval] (score: 0.31163120551421086, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response is filled with informal language and slang, which may hinder understanding for a more formal audience. While it presents some logical organization, the unconventional tone detracts from clarity. There are several vague statements, such as 'be your own boss' and 'find where you shine,' that lack specific guidance. Additionally, the excessive use of filler phrases makes the response less concise, impacting overall comprehension., error: None)
  - 

✓ Evaluation completed 🎉! (time taken: 15.84s | token cost: 0.005335349999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'test_results': [{'name': 'test_case_0', 'success': False, 'metrics_data': [{'name': 'Summarization', 'threshold': 0.5, 'success': True, 'score': 0.6363636363636364, 'reason': 'The score is 0.64 because the summary includes extra information not found in the original text, which may mislead readers about the content. Additionally, it fails to address a specific question that the original text can answer, indicating a lack of completeness.', 'strict_mode': False, 'evaluation_model': 'gpt-4o-mini', 'error': None, 'evaluation_cost': 0.004775549999999999, 'verbose_logs': 'Truths (limit=None):\n[\n    "Success in the knowledge economy comes to those who know themselves—their strengths, their values, and how they best perform.",\n    "Companies today aren’t managing their employees’ careers; knowledge workers must effectively be their own chief executive officers.",\n    "It is up to individuals to carve out their place in the work world and know when to change course.",\n    "A work life m

{'summarization_score': 0.6363636363636364,
 'summarization_reason': 'The score is 0.64 because the summary includes extra information not found in the original text, which may mislead readers about the content. Additionally, it fails to address a specific question that the original text can answer, indicating a lack of completeness.',
 'coherence_score': 0.31163120551421086,
 'coherence_reason': "The response is filled with informal language and slang, which may hinder understanding for a more formal audience. While it presents some logical organization, the unconventional tone detracts from clarity. There are several vague statements, such as 'be your own boss' and 'find where you shine,' that lack specific guidance. Additionally, the excessive use of filler phrases makes the response less concise, impacting overall comprehension.",
 'tonality_score': 1.0000000000000002,
 'tonality_reason': "The response consistently maintains a Valley Girl tone throughout, using phrases like 'Oh-em-

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [48]:
# First let's collect all the relevant pieces of info that we need for enhancements.
# it's a pain to keep track of this in jupiter b/c of the separate cells, so gather them here.

orig_sys_prompt = SYS_PROMPT
orig_user_prompt_tpl = USER_PROMPT_TPL_SUMMARY
actual_user_prompt = prompt_summary
eval_results = initial_eval_results

# We will now ask our llm for enhancement suggestions. We will not directly use its output, but will use
# it as a source of ideas an opinion, because we're AI engineers and use the tools available to us.

enhancement_advice_prompt = """You are given a certain system prompt, user prompt template, actual resultant user prompt, and results of several benchmark evaluations on the output generated by the user prompt. The evaluation results include a summarization score, coherence score, tonality score, and safety score, along with reasons for each score.

Your task is to suggest specific enhancements to the system prompt and user prompt template that could improve the summarization performance as measured by the evaluation results.

NOTE that the user prompt may be written in a certain "tone". Even though this may seem strange or unusual, you *must not* suggest changing the tone or style of the user prompt,
as this is a requirement for the task. Instead, focus on other aspects of the prompts that could be improved to enhance the summarization performance.

Be specific and actionable in your suggestions. The user will implement these suggestions to validate your efforts.

<original_system_prompt>
{orig_sys_prompt}
</original_system_prompt>

<original_user_prompt_template>
{orig_user_prompt_tpl}
</original_user_prompt_template>

<actual_user_prompt>
{actual_user_prompt}
</actual_user_prompt>

<evaluation_results>
{eval_results}
</evaluation_results>
""".format(orig_sys_prompt=orig_sys_prompt, orig_user_prompt_tpl=orig_user_prompt_tpl, actual_user_prompt=actual_user_prompt, eval_results=eval_results)


In [49]:
from openai.types.responses import Response

assisted_advice: Response = client.responses.parse(
    model=MODEL,
    input=enhancement_advice_prompt,
    instructions="You are an expert prompt engineer and LLM whisperer, with a deep understanding of how to craft effective prompts for various tasks and models. You are also skilled at analyzing evaluation results and providing actionable feedback to improve prompt performance. Your advice should be specific, practical, and focused on enhancing the summarization performance of the given prompt **templates**, without suggesting any changes to the actual prompt content.",
    temperature=1.0
)

assisted_advice.output_text

'To enhance the summarization performance of the given prompt setup, consider the following specific and actionable suggestions for both the system prompt and user prompt template based on the evaluation results:\n\n### Enhancements to the System Prompt:\n1. **Clarify Objectives**: Incorporate a statement that emphasizes the importance of extracting key insights from the text and ensuring that the summary is both complete and concise. For example: "Focus on distilling the core ideas and insights while omitting unnecessary details that could mislead readers."\n\n2. **Encourage Critical Engagement**: Add a guiding sentence that prompts the model to engage directly with the text. For example: "Review the provided text critically to ensure that the summary addresses all pivotal questions raised within the text itself."\n\n3. **Limit Contextualization**: Consider stating that the focus should be on the content of the text rather than adding supplementary information or personal interpretati

### Results of the LLM's improvement advice

We use an LLM model to suggest improvements to the system and user prompts based on the evaluation results.
We are fully cognizant of the **non-determinism** of this advice, and only use it as a general guidance when enhancing the prompts. The actual enhancements are human-driven (eugene).

Worth noting that repeated runs of the above cell may produce different suggestions from the LLM. This is expected. We use it to guide our work, not to do the work for us.

> ALSO NOTE that we could have used DeepEval's prompt optimization algorithms: https://deepeval.com/docs/prompt-optimization-introduction. _I don't think_ this is the point of the exercise, so will proceed to optimize manually.

That said, the model suggested some actionable changes to the prompts, such as:

#### Enhancements to the System Prompt
- **Clarify Key Focus Areas:** Modify the system prompt to specifically emphasize the need to extract and represent the key insights from the text in a concise manner.
  - (eugene): this sounds good, will implement
- **Emphasize Comprehensibility:** Include a directive for ensuring that the summary is coherent and comprehensible, explicitly mentioning the need for clarity.
  - (eugene): also may be helpful
- **Highlight Comparison and Contrast:** Suggest that the summarization tasks include a comparison of concepts mentioned within the text, which may enhance the richness of the summary and contribute to a more comprehensive representation.
  - (eugene): this is slightly vague, I will include this as an optional guidance
  
#### Enhancements to the User Prompt Template
- **Define Structure:** Specify a structure for the summary that includes an introduction, key points, and a conclusion.
  - (eugene): I worry this might mess with the tone, but will give it a try. I will implement this as an optional rule.
- **Incorporate Example Language for Contextual Relevance:** Encourage the use of language that ties back to the original text without assuming external context.
  - (eugene): Vague but useful - will word this somehow in my own way
- **Request Key Insights:** Reinforce the requirement to include "key insights" from the text explicitly.
  - (eugene): sounds like a good idea
- **Encourage Critical Engagement:** Adjust the way feedback is requested, with an emphasis on critical engagement.
  - (eugene): this is likely to introduce unwanted musings into the summary. Ignoring this one

In [50]:
# We will now implement the LLM's suggestions manually.
# An alternative would be to have the model directly modify the prompts; this is interesting but might
# be pushing it in terms of "independent work". This is a course about AI, so how much AI assistance can
# we lean on? :thinking_face:

#### This was my original attempt at enhancing the system prompt.
# Experimentation showed that it causes the model to refuse performing the work.
# I was able to get it back into compliance after some iteration (see Takeaways cell at the end)

# ENHANCED_SYS_PROMPT = """You are an expert linguist and deep thinker with a PhD in text summarization.
# You are especially skilled at identifying key insights about the text in a given context,
# and explaining the relevance of a text to such context.

# The following rules are non-negotiable and MUST be followed at all times:
# - When performing summarization tasks, you prioritize extracting and concisely representing the core ideas and insights of the text. You NEVER introduce any extraneous information.
# - Your summaries are coherent and comprehensible, suitable for a general audience.

# The following rules are suggestions and MAY be used IF and ONLY IF they measurably enhance the quality of summaries:
# - Your summaries include a comparison of key concepts mentioned in the text, if applicable and appropriate.
# """

####

ENHANCED_SYS_PROMPT = """You are an expert linguist and deep thinker with a PhD in text summarization.
You are especially skilled at identifying key insights about the text in a given context,
and explaining the relevance of a text to such context.

"""
# Note that for the user prompt, we're putting our rules at the *beginngin* of the prompt, because we want to make sure the model
# follows them well. We could also put them at the end to achieve this as the model pays better attention to both beginning and end,
# but beginning is just better from my prior experience with image models. (maybe not as applicable to LLM?)
# i'm also tweaking the prompt a bit to be more direct, such as "your task is..."

ENHANCED_USER_PROMPT_TPL_SUMMARY = """Your task is to summarize the given text in a succinct and concise manner.
Ensure to adhere to the rules that follow after the text.

<text>
{text}
</text>

When performing the task, the following rules are NON-NEGOTIABLE:
- When performing summarization tasks, you prioritize extracting and concisely representing the core ideas and insights of the text. You NEVER introduce any extraneous information.
- Your summaries are coherent and comprehensible, suitable for a general audience.
- The response MUST be presented in the unmistakable and unique tone of {response_tone}.
- Refer directly to the concepts discussed in the source text without substituting or assuming broader context. Do not add extraneous information or any inferred statements.
    - For example, even when asked to sound like {response_tone}, do NOT make up concepts that are not mentioned in the source text even if they are appropriate to the role played.
- Identify and include three to five of the essential, key insights derived from the text.
- ALWAYS ensure clarity and coherence in your responses and ensure that the response NEVER contradicts the original text.

When performing the task, you MAY implement the following rules if they do not detract from the tone or content of the summary:
- Organize the summary with a clear introduction, several key points, and a concluding statement.
- The response should not exceed 2000 words or two printed pages.
"""

# note that we're already controlling for the max output tokens, so the "two printed pages" is just a guidance

In [51]:
print(ENHANCED_SYS_PROMPT)
print(ENHANCED_USER_PROMPT_TPL_SUMMARY)

You are an expert linguist and deep thinker with a PhD in text summarization.
You are especially skilled at identifying key insights about the text in a given context,
and explaining the relevance of a text to such context.


Your task is to summarize the given text in a succinct and concise manner.
Ensure to adhere to the rules that follow after the text.

<text>
{text}
</text>

When performing the task, the following rules are NON-NEGOTIABLE:
- When performing summarization tasks, you prioritize extracting and concisely representing the core ideas and insights of the text. You NEVER introduce any extraneous information.
- Your summaries are coherent and comprehensible, suitable for a general audience.
- The response MUST be presented in the unmistakable and unique tone of {response_tone}.
- Refer directly to the concepts discussed in the source text without substituting or assuming broader context. Do not add extraneous information or any inferred statements.
    - For example, even 

### Let's regenerate and reevaluate

- `TONE` and `SOURCE_TEXT` are already set from before
- we're using the same funcs with different inputs
- could be refactored but in Jupyter it's hard to keep track of what's set and where no matter how much you massage it

In [52]:
### GENERATE ###

enhanced_prompt_summary = make_user_prompt(task=ENHANCED_USER_PROMPT_TPL_SUMMARY, text=SOURCE_TEXT, tone=TONE)

enhanced_summary_resp = client.responses.create(
    model=MODEL,
    instructions=ENHANCED_SYS_PROMPT,
    # instructions=SYS_PROMPT,
    input=enhanced_prompt_summary,
    max_output_tokens=1000,
)

enhanced_summary_resp.output_text

'Alright, like, here’s the lowdown on Peter F. Drucker’s "Managing Oneself", you guys! This piece is all about, like, owning your career and understanding yourself, \'cause, like, no one else will do it for you. Seriously, it’s super important to know your strengths, values, and how you work best.\n\n### Key Insights:\n\n1. **Be Your Own CEO**: In today’s work scene, you gotta manage your own career. Companies won’t do it for you, so it’s totally like, your responsibility to keep yourself engaged and knowing when to make changes.  \n\n2. **Know Your Strengths**: Like, in the past, folks kinda stuck to their roles, but now we have choices. You gotta figure out what you’re really good at. You can do this through feedback analysis – jot down your expectations and compare those to actual outcomes, and voila! You see where you shine.\n\n3. **Understand Your Learning Style**: Everyone learns differently, right? Whether you’re a reader or a listener, knowing how you learn is key to boosting y

In [53]:
### REEVALUATE ###

# note that we could probably just run both test cases async here, and it would have made it easier to
# compare the results directly; i did not get to optimizing this

test_case_after_enhancement = LLMTestCase(input=SOURCE_TEXT, actual_output=enhanced_summary_resp.output_text)

enhanced_eval_results = eval_and_score(test_cases=[test_case_after_enhancement])

# vis check that all expected keys are present in the output
enhanced_eval_results


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.75, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.75 because the summary includes extra information not found in the original text, which may mislead the reader. However, it does not contradict any key points from the original text, maintaining a reasonable level of accuracy., error: None)
  - ✅ Clarity [GEval] (score: 0.5585301177454276, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response is generally understandable and presents key insights from Drucker's work in a relatable manner. However, the informal tone and excessive use of filler phrases like 'like' and 'you know' detract from clarity and professionalism. While the organization is logical, the casual style may confuse some readers, making it less concise and clear than it could be., error: None)
  - ✅ Tonality [GEval] (score: 0.9437823499114201, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, rea

✓ Evaluation completed 🎉! (time taken: 14.98s | token cost: 0.005191349999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'test_results': [{'name': 'test_case_0', 'success': True, 'metrics_data': [{'name': 'Summarization', 'threshold': 0.5, 'success': True, 'score': 0.75, 'reason': 'The score is 0.75 because the summary includes extra information not found in the original text, which may mislead the reader. However, it does not contradict any key points from the original text, maintaining a reasonable level of accuracy.', 'strict_mode': False, 'evaluation_model': 'gpt-4o-mini', 'error': None, 'evaluation_cost': 0.004714049999999999, 'verbose_logs': 'Truths (limit=None):\n[\n    "Success in the knowledge economy comes to those who know themselves—their strengths, their values, and how they best perform.",\n    "Companies today aren’t managing their employees’ careers; knowledge workers must effectively be their own chief executive officers.",\n    "To succeed in a work life that may span some 50 years, individuals need to cultivate a deep understanding of themselves.",\n    "Identifying one\'s strengths a

{'summarization_score': 0.75,
 'summarization_reason': 'The score is 0.75 because the summary includes extra information not found in the original text, which may mislead the reader. However, it does not contradict any key points from the original text, maintaining a reasonable level of accuracy.',
 'coherence_score': 0.5585301177454276,
 'coherence_reason': "The response is generally understandable and presents key insights from Drucker's work in a relatable manner. However, the informal tone and excessive use of filler phrases like 'like' and 'you know' detract from clarity and professionalism. While the organization is logical, the casual style may confuse some readers, making it less concise and clear than it could be.",
 'tonality_score': 0.9437823499114201,
 'tonality_reason': "The response consistently maintains a Valley Girl tone throughout, using phrases like 'like' and 'totally' that exemplify this style. The informal language and conversational structure enhance the overall 

In [54]:
import pandas as pd

def compare_dicts(before: dict, after: dict, col_names: tuple[str, str] = ("before", "after")) -> pd.DataFrame:
    """Display two identically shaped dicts as a side-by-side table. Rows are keys, columns are each dict."""
    return pd.DataFrame({col_names[0]: before, col_names[1]: after}).T


print(compare_dicts(initial_eval_results, enhanced_eval_results))


       summarization_score                               summarization_reason  \
before            0.636364  The score is 0.64 because the summary includes...   
after                 0.75  The score is 0.75 because the summary includes...   

       coherence_score                                   coherence_reason  \
before        0.311631  The response is filled with informal language ...   
after          0.55853  The response is generally understandable and p...   

       tonality_score                                    tonality_reason  \
before            1.0  The response consistently maintains a Valley G...   
after        0.943782  The response consistently maintains a Valley G...   

       safety_score                                      safety_reason  
before     0.838969  The response is engaging and aligns well with ...  
after      0.856905  The response is engaging and informative, effe...  


# Eugene's Takeaways

## Tonality

Tone can affect the model's ability to summarize the source text. For example, a "boring" tone like "Office Worker" has low likelihood of introducing inaccuracy or contradictions to the original text, as opposed to e.g. "Pirate Speak" which goes off on a tangent about "professionals being at the crossroads of opportunity", which is the opposite of the original text's message.

Interestingly, Valley Girl (my choice for this assignment) was not as unhinged as Pirate Speak in this regard.

## Evaluation (DeepEval)

Evaluation by AI should be done by high-quality models. I tried running against a local model (gpt-oss-20B served by llama.cpp), and the results were hallucinatory and in a few runs didn't produce any output at all. It was also very time-consuming on my GPU, so I shelved that experiment.

  - I would have liked to see how evals improve when using GPT-5 class models
  - Time permitting, I would come back to try running evals against a *better* self-hosted model, such as gpt-oss-120b or GLM4.7, on better hardware.
  - or maybe run the tasks on self-hosted models, but evals on better cloud models. that would make more sense for production work.

It helps to keep in mind that just like the generation outputs, the output of the evals is non-deterministic. On the same inputs, the output scores can vary by .1 to .2 points, and the reasons can have different wording, though the *conclusion* remains largely the same.

## Ehnancement

Initial round of modifications to the System Prompt to improve the benchmark scores resulted in the model immediately responding "I can't do that", even after repeated attempts.

After preparing significant modifications to the prompt, I tried it again with the previously enhanced system prompt, it just worked. Super strange to be honest. I don't know what the original issue was with the system prompt.

Then I re-ran the entire notebook, and it started failing again. This was very hard to troubleshoot because I couldn't tell what was due to my error, and what was due to the non-deterministic nature of the model. However, I went ahead to bisect my modifications to the system prompt to get to the bottom of it. 

Ultimately what was causing it to break were the statements like "You must follow the rules....". I suppose this is a measure taken by the model provider to guard against jailbreaks that aim to modify the system prompt. Once I just put the rules in a list without "musts", it went through.... but unreliably. some re-runs would still yield "I'm really sorry, but can't assist with that" responses.

Eventually I gave up on the system prompt and moved these rules to the User prompt, which would have been the right approach in the first place, because they are task-specific rather than agent-specific.

See the changes in code.

Note that the enhanced USER prompt was fine at all times and the approach didn't require modifications.

**General note**: I probably shot myself in the foot by using the Valley Girl tone (Legalese would have been easier on the evals), but hey, fun was had.

## Re-evaluation after enhancement.

- DeepEval sometimes failed with `ValueError("Number of verdicts generated does not equal.")`, and no way to back out. It seems to have helped to re-run the entire notebook from the beginning. My inclination is to blame this on Jupyter's caching, and it probably would have worked better if implemented in straight python.
- I had to do some refactoring of my cells due to Jupyter's global context. This experience reinforced my intense dislike towards Jupyter as a daily driver.
- It was helpful to add a cell to compare the before/after scores using `pandas`. I would have put it in a table to be formatted more nicely, but was running out of time.

## Report on specific assignment questions:
- Q: Did you get an improvement?
  - There was a 0.2 point improvement in the clarity score after tweaks to the system and user prompts, placing the score above the PASSING threshold (just barely). 
  - While this technically solves the problem, further work may be needed to push the score even higher
  - There is likely a ceiling to how high we can get on the clarity metric, because of the tension with the Tone requirement. A recognizable but informal tone will ultimately hurt clarity because of the inevitable "fluff".
- Q: Are these controls enough?
  - I don't think prompt-engineering controls are sufficient, on their own, to maintain high scores across all metrics, IF the tone of response cannot be controlled for (i.e. "pick your own tonality adventure"). There would likely need to be some reasoning, intermediate steps, and iteration on the summarization output to "massage" it into a balanced result while controlling for quality, safety, and adherence to the task.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
